# v11 — satu model tim (notebook yang sama untuk keenam model)

Model ditentukan oleh baris `MODEL` di sel **Setelan**. Panduan lengkap tim: `../README.md`.

**Cara pakai**
1. Ubah hanya sel **Setelan**. `SETELAN` untuk mencoba setelan lain, dan **ganti `NAMA_SETELAN` setiap kali `SETELAN` berubah**, supaya hasil lama tidak tertimpa (kalau lupa, notebook berhenti dengan pesan).
2. *Run All*. Training 5 ulangan × 5 fold pasien (protokol v3) bisa dilanjutkan: kalau sesi Kaggle/Colab terputus, jalankan lagi; fold yang sudah selesai dilewati.
3. Isi sel **Kesimpulan** dengan angka dari tabel, lalu bagikan zip hasil (sel terakhir) ke tim.

`SIHEDAF_QUICK=1` hanya untuk uji alur (data kecil, 1 epoch). Hasilnya **bukan** hasil penelitian dan disimpan terpisah (`cache/v11_quick/`).

In [1]:
# ── Setelan: satu-satunya sel yang perlu diubah ────────────────────────────────
MODEL = "inceptiontime"        # nama folder model (jangan diubah)
NAMA_SETELAN = "bawaan"    # GANTI setiap kali SETELAN diubah, mis. "percobaan2"
SETELAN = {}               # menimpa setelan bawaan model.py, mis. {"epochs": 40}; kunci yang ada dicetak di bagian 1
#   balancing (opsional, hanya window latih tiap fold): {"balancing": "under"} atau "over" untuk semua model,
#   {"balancing": "smote"} hanya svm/xgboost. Bawaan None = tanpa balancing (lihat ../README.md)
HASIL_DI = None            # Colab: mis. "/content/drive/MyDrive/sihedaf_v11" agar hasil tidak hilang (lihat README)
DATA_DI = None             # isi hanya bila unduhan data gagal: folder hasil ekstrak v11_data.zip (lihat README)
REPO_URL = "https://github.com/farhanlhsn/WrapSihedaf.git"
BRANCH = "main"            # pakai fork/branch sendiri bila mengubah arsitektur di model.py
DATA_URL = "https://github.com/farhanlhsn/WrapSihedaf/releases/download/v11-data/v11_data.zip"

import os
import subprocess
import sys
from pathlib import Path

if HASIL_DI and str(HASIL_DI).startswith("/content/drive") and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive                      # Colab: sambungkan Google Drive
    drive.mount("/content/drive")
if HASIL_DI:
    os.environ["SIHEDAF_HASIL"] = str(HASIL_DI)
if DATA_DI:
    os.environ["SIHEDAF_DATA"] = str(DATA_DI)
if not Path("model.py").exists():                       # Kaggle/Colab: clone repo dulu
    root = Path("/kaggle/working/WrapSihedaf") if Path("/kaggle/working").exists() else Path.cwd() / "WrapSihedaf"
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(root)], check=True)
    os.chdir(root / "v11_model_tim" / MODEL)
    req = Path(f"requirements-{MODEL}.txt")
    if req.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=True)
assert Path.cwd().name == MODEL, f"cwd harus folder v11_model_tim/{MODEL}, sekarang {Path.cwd()}"
for p in (str(Path.cwd().parent), str(Path.cwd())):
    if p not in sys.path:
        sys.path.insert(0, p)

from bersama import data, evaluasi, runner

data.siapkan_data(DATA_URL)
import torch

print("cwd   :", Path.cwd())
print("commit:", subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())
print("GPU   :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada")
print("data  :", data.lokasi_data())
print("hasil :", data.lokasi_hasil())
print("QUICK :", data.QUICK, "(uji alur, BUKAN hasil)" if data.QUICK else "")

cwd   : /mnt/HDD/03_Akademik_dan_Tugas/WrapSihedaf/v11_model_tim/inceptiontime
commit: b18d6bc docs: tegaskan hanya paket v3.1 yang siap pakai untuk backend & uji perangkat
GPU   : NVIDIA GeForce RTX 5060
data  : /mnt/HDD/03_Akademik_dan_Tugas/WrapSihedaf/cache
hasil : /mnt/HDD/03_Akademik_dan_Tugas/WrapSihedaf/cache/v11
QUICK : False 


## 1. Data dan model

In [2]:
import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", None)   # teks vonis tidak terpotong

import model as MDL

d = data.muat()
print(f"{len(d.y):,} window uji detektor, {len(d.patients)} pasien, fold {d.folds.shape} (ulangan × window)")
print("Setelan bawaan:", {**runner.SETELAN_BERSAMA, **MDL.SETELAN})
print("Ditimpa       :", SETELAN)

165,978 window uji detektor, 23 pasien, fold (5, 165978) (ulangan × window)
Setelan bawaan: {'balancing': None, 'filter': 16, 'kernel': (3, 5, 9), 'kedalaman': 6, 'dropout': 0.0, 'epochs': 30, 'batch_size': 256, 'lr': 0.001, 'weight_decay': 0.0001, 'weight_power': 0.5}
Ditimpa       : {}


## 2. Training 5 ulangan × 5 fold pasien

Bisa dilanjutkan. Bobot model, prediksi fold uji, dan prediksi uji stres tiap fold disimpan di `cache/v11/<model>/<NAMA_SETELAN>/`.

In [3]:
folder = runner.jalankan(MDL, NAMA_SETELAN, SETELAN)
print("Hasil:", folder)

[   1.8 mnt] inceptiontime/bawaan ulangan 0 fold 0: latih 108 s, window latih per kelas [82012, 46765, 1143]


[   3.7 mnt] inceptiontime/bawaan ulangan 0 fold 1: latih 111 s, window latih per kelas [79227, 56537, 1106]


[   5.5 mnt] inceptiontime/bawaan ulangan 0 fold 2: latih 110 s, window latih per kelas [79218, 56957, 846]


[   7.3 mnt] inceptiontime/bawaan ulangan 0 fold 3: latih 106 s, window latih per kelas [73927, 54733, 1055]


[   8.9 mnt] inceptiontime/bawaan ulangan 0 fold 4: latih 100 s, window latih per kelas [81616, 48312, 458]


[  10.6 mnt] inceptiontime/bawaan ulangan 1 fold 0: latih 99 s, window latih per kelas [81219, 47346, 1122]


... (15 outputs left out)

[  38.9 mnt] inceptiontime/bawaan ulangan 4 fold 1: latih 109 s, window latih per kelas [85586, 51070, 479]


[  40.7 mnt] inceptiontime/bawaan ulangan 4 fold 2: latih 108 s, window latih per kelas [75721, 60139, 1056]


[  42.4 mnt] inceptiontime/bawaan ulangan 4 fold 3: latih 103 s, window latih per kelas [79135, 49334, 837]


[  44.2 mnt] inceptiontime/bawaan ulangan 4 fold 4: latih 104 s, window latih per kelas [80938, 48635, 1130]
Hasil: /mnt/HDD/03_Akademik_dan_Tugas/WrapSihedaf/cache/v11/inceptiontime/bawaan


## 3. Evaluasi (protokol v3)

Macro-F1 pooled OOF rata-rata ulangan + CI 95% bootstrap pasien. `+ bias` = bias kelas cross-fit (grid −7…+3). Pembanding v1 dan v3.1 memakai ulangan dan sampel bootstrap yang sama. Vonis: P ≥ 0,95 lebih baik, ≤ 0,05 lebih buruk, selebihnya tidak terbedakan.

In [4]:
r = evaluasi.laporan_model(folder, d)
label = {"v1": "v1 (argmax)", "v31": "v3.1 (referensi)", "argmax": f"{MODEL} argmax", "bias": f"{MODEL} + bias"}
fmt = lambda v: f"{v[0]:.3f} [{v[1]:.2f}–{v[2]:.2f}]"
kolom = ["Macro-F1", "AFL F1", "AFL P", "AFL R", "AFIB F1", "Normal R"]
print(f"Ulangan yang lengkap: {r['ulangan']}")
display(pd.DataFrame({label[k]: {m: fmt(r["metrik"][k][m]) for m in kolom} for k in label}).T)
display(pd.DataFrame({
    f"{MODEL} + bias vs {label[ref]}": {
        m: f"Δ {v[0]:+.3f} [{v[1]:+.2f}…{v[2]:+.2f}], P = {v[3]:.2f} → {evaluasi.vonis(v[3])}"
        for m, v in r[f"banding_{ref}"].items()}
    for ref in ("v31", "v1")}))

Ulangan yang lengkap: [0, 1, 2, 3, 4]


,Macro-F1,AFL F1,AFL P,AFL R,AFIB F1,Normal R
v1 (argmax),0.744 [0.64–0.81],0.399 [0.09–0.59],0.275 [0.05–0.46],0.728 [0.37–0.98],0.898 [0.85–0.93],0.903 [0.82–0.95]
v3.1 (referensi),0.832 [0.67–0.92],0.647 [0.17–0.89],0.715 [0.26–0.95],0.592 [0.13–0.91],0.909 [0.86–0.94],0.924 [0.84–0.97]
inceptiontime argmax,0.746 [0.64–0.90],0.373 [0.05–0.82],0.281 [0.03–0.78],0.657 [0.19–0.95],0.926 [0.90–0.95],0.915 [0.85–0.96]
inceptiontime + bias,0.767 [0.64–0.86],0.446 [0.06–0.72],0.441 [0.05–0.85],0.467 [0.08–0.74],0.914 [0.89–0.94],0.947 [0.91–0.98]


,inceptiontime + bias vs v3.1 (referensi),inceptiontime + bias vs v1 (argmax)
Macro-F1,"Δ -0.065 [-0.12…-0.02], P = 0.01 → lebih buruk","Δ +0.023 [-0.01…+0.09], P = 0.86 → tidak terbedakan"
AFL F1,"Δ -0.201 [-0.35…-0.08], P = 0.00 → lebih buruk","Δ +0.048 [-0.04…+0.24], P = 0.80 → tidak terbedakan"
AFL P,"Δ -0.274 [-0.39…+0.04], P = 0.13 → tidak terbedakan","Δ +0.166 [+0.01…+0.56], P = 0.98 → lebih baik"
AFL R,"Δ -0.124 [-0.19…-0.05], P = 0.01 → lebih buruk","Δ -0.261 [-0.37…-0.22], P = 0.00 → lebih buruk"
Normal R,"Δ +0.023 [-0.01…+0.07], P = 0.93 → tidak terbedakan","Δ +0.044 [+0.02…+0.09], P = 1.00 → lebih baik"


In [5]:
display(pd.DataFrame({label[k]: v for k, v in r["afl_per_pasien"].items()}).round(2).rename_axis("recall AFL per pasien"))
display(pd.DataFrame({label[k]: v for k, v in r["afl_palsu"].items()}).round(0).rename_axis("AFL palsu (window/ulangan)"))
b = r["bias"]
print(f"Bias [AFIB, AFL] median {b['median'][1:]}, rentang {b['min'][1:]} … {b['maks'][1:]}")
if b["di_batas"] > 0:
    print(f"⚠ {b['di_batas']:.0%} bias menyentuh batas grid (−7/+3): tanda bahaya (AGENTS.md, pelajaran v2/v9).")

,v1 (argmax),v3.1 (referensi),inceptiontime argmax,inceptiontime + bias
recall AFL per pasien,,,,
04043,1.00,0.40,0.80,0.80
04908,1.00,0.92,0.86,0.60
04936,0.67,0.51,0.59,0.40
06426,0.73,0.17,0.35,0.07
07910,0.21,0.00,0.05,0.01
08215,0.04,0.00,0.00,0.00
08378,1.00,0.96,1.00,0.79


,v1 (argmax),v3.1 (referensi),inceptiontime argmax,inceptiontime + bias
AFL palsu (window/ulangan),,,,
total,2240.0,274.0,2470.0,937.0
dari_normal,521.0,55.0,2112.0,783.0
dari_afib,1719.0,219.0,358.0,155.0
pasien_tanpa_afl,1059.0,25.0,155.0,36.0


Bias [AFIB, AFL] median [-1.75, -4.0], rentang [-4.25, -6.75] … [0.0, -1.75]


## 4. Uji stres takikardia (protokol butir 9)

Ritme teratur sintetis 60–170 bpm × 4 jenis variasi detak. **Ambang** = detak terendah di mana < 50% window ditebak Normal (makin tinggi makin baik). Pembanding: kontrol MicroTransformer per fold dari v7 dan paket produksi v3.1 (115 bpm).

In [6]:
tk = r["takikardia"]
display(pd.DataFrame({MODEL: tk["model"]["ambang"], "kontrol v7 (per fold)": tk["kontrol_v7"]["ambang"],
                      "paket v3.1": {j: tk["paket_v31"] for j in tk["model"]["ambang"]}}).rename_axis("ambang (bpm)"))
normal = pd.DataFrame({j: [p[0] for p in tab] for j, tab in tk["model"]["proporsi"].items()}, index=tk["model"]["bpm"])
display((normal * 100).round(0).T.rename_axis(f"% ditebak Normal oleh {MODEL}"))
pita = {k: {"window": v["n"], "pasien": v["n_pasien"],
            "v3.1: % AFIB/AFL": 100 * (v["v31"][1] + v["v31"][2]),
            f"{MODEL}: % AFIB/AFL": 100 * (v["model"][1] + v["model"][2])}
        for k, v in r["pita_detak"].items() if k.startswith("Normal")}
display(pd.DataFrame(pita).T.round(1).rename_axis("window Normal asli per pita detak"))

,inceptiontime,kontrol v7 (per fold),paket v3.1
ambang (bpm),,,
tanpa variasi,120,120,115
seragam ±10 ms,130,120,115
Gauss sd 10 ms,130,120,115
Gauss sd 20 ms,125,115,115


,60,65,70,75,80,85,90,95,100,105,...,125,130,135,140,145,150,155,160,165,170
% ditebak Normal oleh inceptiontime,,,,,,,,,,,,,,,,,,,,,
tanpa variasi,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,...,8.0,4.0,4.0,4.0,4.0,0.0,0.0,0.0,0.0,0.0
seragam ±10 ms,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,99.0,...,54.0,23.0,6.0,4.0,4.0,0.0,0.0,0.0,0.0,0.0
Gauss sd 10 ms,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,98.0,...,66.0,35.0,12.0,4.0,4.0,0.0,0.0,0.0,0.0,0.0
Gauss sd 20 ms,100.0,100.0,100.0,100.0,100.0,100.0,99.0,98.0,96.0,94.0,...,49.0,28.0,17.0,12.0,8.0,6.0,5.0,3.0,1.0,1.0


,window,pasien,v3.1: % AFIB/AFL,inceptiontime: % AFIB/AFL
window Normal asli per pita detak,,,,
Normal | 0–100 bpm,89566.0,21.0,6.7,4.1
Normal | 100–110 bpm,8766.0,12.0,12.7,13.7
Normal | 110–120 bpm,568.0,10.0,59.0,44.6
Normal | 120–∞ bpm,100.0,7.0,95.2,72.2


## 5. Kesimpulan

Setelan `bawaan`, 5 ulangan lengkap, PC dengan GPU RTX 5060 (106 s per fold, notebook 47 menit, 3 November 2026). Rincian: `README.md`.

- **Macro-F1 + bias 0,767 [0,64–0,86]: lebih buruk daripada v3.1** (Δ −0,065, P = 0,006), setara v1 (+0,023, P = 0,86). Bukan kandidat pengganti paket produksi.
- **AFL F1 0,446 [0,06–0,72]**, recall AFL 0,467 (v3.1 0,592, P = 0,005). Tanpa bias 08378 terdeteksi penuh (1,00), tetapi AFL palsu 2.470 window per ulangan. Bias AFL (median −4) menekannya ke 937 (3,4× v3.1) dan ikut menurunkan 08378 ke 0,79: pola TCN bawaan, lebih ringan.
- **Recall AFL per pasien (+ bias):** 04043 0,80 · 04908 0,60 · 04936 0,40 · 06426 0,07 · 07910 0,01 · 08215 0,00 · 08378 0,79.
- **Bias** [AFIB, AFL] median −1,75 / −4; tidak ada yang menyentuh batas grid, tetapi AFL terendah −6,75 sudah dekat −7.
- **Takikardia:** ambang 120 / 130 / 130 / 125 bpm (v3.1 115). Alarm pada Normal ≥ 120 bpm 72,2% (v3.1 95,2%). Normal R 0,947 (v3.1 0,924). Trade-off dari bias yang kuat.
- Bobot kelas penuh (`{"weight_power": 1.0}`, setelan `wp100`) sudah dicoba 3 November 2026 lewat `eksperimen.py e2`. 08378 pulih (0,99), tetapi Macro-F1 tidak berubah (0,766, P = 0,53 vs bawaan), karena AFL palsu ikut naik dan bias AFL menempel di −7 pada 8 dari 25 fold. Lihat `README.md` "Hasil 2".

In [7]:
print(f"Latih rata-rata {r['waktu_per_fold']:.0f} s per fold | info model: {r['info']}")
print("Zip hasil untuk dibagikan ke tim:", runner.zip_hasil(folder))

Latih rata-rata 106 s per fold | info model: {'parameter': 41859, 'n_latih': [82012, 46765, 1143]}


Zip hasil untuk dibagikan ke tim: /mnt/HDD/03_Akademik_dan_Tugas/WrapSihedaf/cache/v11/inceptiontime/inceptiontime_bawaan.zip
